In [1]:
from pyspark.sql import functions as F

df_watermark = spark.read.table("control_watermark")

df_watermark.show(truncate=False)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 3, Finished, Available, Finished, False)

+-----------+------------------------+-------------------+
|source_name|table_name              |last_watermark     |
+-----------+------------------------+-------------------+
|SEO        |Fact_Keyword_Performance|2026-01-31 23:50:00|
|SEO        |Fact_SEO_Traffic        |2026-01-31 23:54:00|
|SEO        |Fact_SEO_Orders         |2026-01-31 22:38:00|
+-----------+------------------------+-------------------+



In [2]:
traffic_watermark = (
    df_watermark
    .filter(F.col("table_name") == "Fact_SEO_Traffic")
    .select("last_watermark")
    .first()["last_watermark"]
)

print("Traffic watermark:", traffic_watermark)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 4, Finished, Available, Finished, False)

Traffic watermark: 2026-01-31 23:54:00


In [3]:
keyword_watermark = (
    df_watermark
    .filter(F.col("table_name") == "Fact_Keyword_Performance")
    .select("last_watermark")
    .first()["last_watermark"]
)

print("Keyword watermark:", keyword_watermark)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 5, Finished, Available, Finished, False)

Keyword watermark: 2026-01-31 23:50:00


In [4]:
orders_watermark = (
    df_watermark
    .filter(F.col("table_name") == "Fact_SEO_Orders")
    .select("last_watermark")
    .first()["last_watermark"]
)

print("Orders watermark:", orders_watermark)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 6, Finished, Available, Finished, False)

Orders watermark: 2026-01-31 22:38:00


In [8]:
df_orders_batch02 = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/incremental/batch_02/seo_orders/seo_orders_batch_02.csv")
)

print("Batch 02 Orders:", df_orders_batch02.count())

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 10, Finished, Available, Finished, False)

Batch 02 Orders: 3298


In [9]:
from pyspark.sql import functions as F

orders_watermark = (
    spark.read.table("control_watermark")
    .filter(F.col("table_name") == "Fact_SEO_Orders")
    .select("last_watermark")
    .first()["last_watermark"]
)

print("Orders watermark:", orders_watermark)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 11, Finished, Available, Finished, False)

Orders watermark: 2026-01-31 22:38:00


In [10]:
df_orders_incremental = df_orders_batch02.filter(
    F.col("updated_timestamp") > F.lit(orders_watermark)
)

print("Incremental Orders:", df_orders_incremental.count())

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 12, Finished, Available, Finished, False)

Incremental Orders: 3298


In [11]:
df_orders_incremental.select(
    F.min("updated_timestamp").alias("min_updated_timestamp"),
    F.max("updated_timestamp").alias("max_updated_timestamp")
).show(truncate=False)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 13, Finished, Available, Finished, False)

+---------------------+---------------------+
|min_updated_timestamp|max_updated_timestamp|
+---------------------+---------------------+
|2026-02-01 03:39:00  |2026-05-31 23:51:00  |
+---------------------+---------------------+



In [12]:
# ==============================
# Batch 02 - Traffic
# ==============================

df_traffic_batch02 = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(
        "abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/incremental/batch_02/seo_traffic/seo_traffic_batch_02.csv"
    )
)

traffic_watermark = (
    spark.read.table("control_watermark")
    .filter(F.col("table_name") == "Fact_SEO_Traffic")
    .select("last_watermark")
    .first()["last_watermark"]
)

df_traffic_incremental = df_traffic_batch02.filter(
    F.col("updated_timestamp") > F.lit(traffic_watermark)
)

print("Traffic Batch 02 rows:", df_traffic_batch02.count())
print("Traffic incremental rows:", df_traffic_incremental.count())

df_traffic_incremental.select(
    F.min("updated_timestamp").alias("min_updated_timestamp"),
    F.max("updated_timestamp").alias("max_updated_timestamp")
).show(truncate=False)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 14, Finished, Available, Finished, False)

Traffic Batch 02 rows: 16672
Traffic incremental rows: 16672
+---------------------+---------------------+
|min_updated_timestamp|max_updated_timestamp|
+---------------------+---------------------+
|2026-02-01 00:05:00  |2026-05-31 23:27:00  |
+---------------------+---------------------+



In [13]:
# ==============================
# Batch 02 - Keyword Performance
# ==============================

df_keyword_batch02 = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(
        "abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Bronze.Lakehouse/Files/bronze/incremental/batch_02/seo_keyword_performance/seo_keyword_performance_batch_02.csv"
    )
)

keyword_watermark = (
    spark.read.table("control_watermark")
    .filter(F.col("table_name") == "Fact_Keyword_Performance")
    .select("last_watermark")
    .first()["last_watermark"]
)

df_keyword_incremental = df_keyword_batch02.filter(
    F.col("updated_timestamp") > F.lit(keyword_watermark)
)

print("Keyword Batch 02 rows:", df_keyword_batch02.count())
print("Keyword incremental rows:", df_keyword_incremental.count())

df_keyword_incremental.select(
    F.min("updated_timestamp").alias("min_updated_timestamp"),
    F.max("updated_timestamp").alias("max_updated_timestamp")
).show(truncate=False)

StatementMeta(, d5307f86-5689-43c3-8825-e021dbbb7641, 15, Finished, Available, Finished, False)

Keyword Batch 02 rows: 9840
Keyword incremental rows: 9840
+---------------------+---------------------+
|min_updated_timestamp|max_updated_timestamp|
+---------------------+---------------------+
|2026-02-01 00:03:00  |2026-05-31 23:52:00  |
+---------------------+---------------------+

